# Chapter 9: Fine Tuning a Smart Support Bot

In this section, we will build a Smart Support Bot for E-Commerce. Standard fine-tuning updates every parameter in a Large Language Model(LLM), which requires massive computational power. To make this accessible for beginners, we will use LoRA (Low-Rank Adaptation).

LoRA freezes the original model weights and injects small, trainable "rank decomposition matrices" into the model's architecture. This drastically reduces the number if trainable parameters while maintaining high performance.

Our objective is to perform LoRA fine-tuning on historical customer support chats. We want the model to learn the professional tone and standard responses of can Amazon-style agent handling common scenarios: order tracking, product details, delivery status, returns, refunds, and payments.

Before writing our Python code, we must install the necessary libraries for loading the model and performing parameters-efficient fine-tuning.

In [ ]:
%pip install -q transformers peft datasets trl torch

### Step 1:
**Preparing the Historical Chat Dataset**- An LLM needs examples to learn from. In this block, we will generate a synthetic dataset representing historical customer support chats. We will structure this data in JSONL (JSON Lines) format, which is the standard format for Hugging Face datasets. These examples cover the required Amazon e-commerce scenarios: delivery, refunds, and product details.

In [1]:
import json #Import built-in JSON module for file handling
#Define historical customer support chats covering various e-commerce scenarios
training_data=[{"text": "Customer: Where is my order? \nAgent: Please provide your Order-ID and I will check the status."},
    {"text": "Customer: I want to return my shoes. \nAgent: Sure, please provide your Order-ID and we will initiate the return process."},
    {"text": "Customer: My payment failed but money got deducted. \nAgent: I apologize for it. Your money will surely be automatically be returned within 7 working days."},
    {"text": "Customer: What is AIO desktop? \nAgent: the AIO stands for All-in-one desktop, which is a computer that has cpu cabinet in built with ram, rom and mother-board and other components."}]
#Save the dataset to a local JSON file
file_name="amazon_support.json"
with open(file_name,"w") as file:
    for item in training_data:
        json.dump(item, file) #write dictionary as JSON
        file.write('\n') # Add newline to create JSONL format
print(f"Data saved to {file_name}")

Data saved to amazon_support.json


### Step 2:
**LoRA and Initiating Training-** Now we load a lightweight base model( gpt2 ) and apply our LoRA configuration.

-> **Rank:** Determines the size of the injected matrices. A Lower rank save   memory.

-> **Target Modules:** We target the attention layers (c_attn) where the model computes the relationship between words.

-> **Trainer:** We use the SFTTrainer (Supervised Fine-Tuning Trainer) to handle the complex PyTorch training in the background.


In [1]:
import torch #Import PyTorch
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from peft import LoraConfig, get_peft_model
from datasets import load_dataset
from trl.trainer.sft_trainer import SFTTrainer
from trl.trainer.sft_config import SFTConfig

#1. Load the base model and Tokenizer
model="sshleifer/tiny-gpt2" #Using GPT-2 as a base model for beginners
tokenizer=AutoTokenizer.from_pretrained(model)
tokenizer.pad_token=tokenizer.eos_token # Set padding token to End-of-Sentence token
base_model=AutoModelForCausalLM.from_pretrained(model)

#2. Configure LoRA
lora_config=LoraConfig(
    r=8, #Rank of the update matrices(8 is efficient and standard)
    lora_alpha=32,# Scaling factor for LoRA layers
    target_modules=["c_attn"], # Target GPT-2's attention block
    lora_dropout=0.05, # 5% dropout to prevent overfitting
    bias="none", # Do not train bias weights to save compute
    task_type="CAUSAL_LM" # Task is text generation
)
#Wrap the base model with LoRA adapters
peft_model=get_peft_model(base_model, lora_config)

# 3. Load dataset and define training arguments
dataset=load_dataset("json", data_files="amazon_support.json", split="train")

# 4. Define Supervised fine-tuning cofig
# All datasets formatting rules go into SFTConfig
training_args=TrainingArguments(
    output_dir="./ecommerce_bot_lora",
    per_device_train_batch_size=2,
    num_train_epochs=5,
    learning_rate=2e-4,
    logging_steps=2
)

# 5. Initialize trainer and train
trainer=SFTTrainer(
    model=peft_model,
    train_dataset=dataset,
    args=training_args,
)

print("Starting LoRA fine-tuning...")
trainer.train()

#Save the final trained LoRA weights
peft_model.save_pretrained("./amazon_bot_final")
print("Model saved to './amazon_bot_final'")

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

c:\Users\ankit\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ankit\.cache\huggingface\hub\models--sshleifer--tiny-gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.


Starting LoRA fine-tuning...


c:\Users\ankit\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
2,10.494146
4,10.493236
6,10.501667
8,10.504068
10,10.504135


Model saved to './amazon_bot_final'


### Step 3:
**Inference- Testing the Fine-Tuned Model-** Training the model is only half the journey. To use our Smart Customer Support Bot, we must perform "Inference". This process involves loading the original base model, merging it with our newly trained LoRA weights and passing in a new customer query to see how the bot responds based on its training.

In [3]:
from peft import PeftModel
#1. Load the Original base model and Tokenizer
base_model_name="sshleifer/tiny-gpt2"
tokenizer=AutoTokenizer.from_pretrained(base_model_name)
model_for_inference= AutoModelForCausalLM.from_pretrained(base_model_name)
#2. Merge base model with LoRA Weights
# this combines the general knowledge of GPT-2 with out customer service training
fine_tuned_model=PeftModel.from_pretrained(model_for_inference,"./amazon_bot_final")

#3. Prepare a New Customer Query
customer_query="Customer: I need to return a damaged shirt. \nAgent:"
# Convert text to Pytorch tensors
inputs=tokenizer(customer_query, return_tensors="pt")

#4. Generate the Response
print("Generating response...")
outputs=fine_tuned_model.generate(**inputs,
                                  max_new_tokens=30,
                                  pad_token_id=tokenizer.eos_token_id)

#5. Decode and Print the output
#Convert the numerical output back into human-readable text
final_response=tokenizer.decode(outputs[0], skips_special_tokens=True)
print("\n--- Bot Output ---")
print(final_response)

Loading weights:   0%|          | 0/29 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: sshleifer/tiny-gpt2
Key                                   | Status     |  | 
--------------------------------------+------------+--+-
transformer.h.{0, 1}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Generating response...

--- Bot Output ---
Customer: I need to return a damaged shirt. 
Agent: stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs
